In [ ]:
import os
import json
import matplotlib.pyplot as plt

# Get the input and output file paths from environment variables
input_path = os.getenv('INPUT_PATH')  # Path to the intermediate JSON file
output_path = os.getenv('OUTPUT_PATH')  # Path to save the final results

# Read the intermediate JSON file
with open(input_path, 'r') as f:
    data = json.load(f)

# Display the data (optional, for debugging)
print("Intermediate Data:", data)

# Ensure years match the "LATA" field in data
years = data.get("LATA", [])
print(f"Years from LATA: {years}")

# Dynamically process all known fields if they exist
fields = [
    "twKI", "twKS", "twINW", "twEKS", "twIMP",
    "KI", "KS", "INW", "EKS", "IMP", "PKB", "ZDEKS", "NET_EXPORTS"
]

# Log missing fields for debugging
for field in fields:
    if field not in data:
        print(f"Warning: Field '{field}' is missing in the input data.")

# Extract existing fields and calculate ZDEKS
PKB = data.get("PKB", [])
EKS = data.get("EKS", [])
IMP = data.get("IMP", [])

# Calculate export-to-GDP ratio (ZDEKS)
ZDEKS = [eks / pkb if pkb != 0 else 0 for eks, pkb in zip(EKS, PKB)]
data["ZDEKS"] = ZDEKS
print("Calculated ZDEKS:", ZDEKS)


# Display the updated data (optional, for debugging)
print("Updated Data with Calculations:", data)

# Separate fields into two groups: large numbers and small numbers
large_fields = ["KI", "KS", "INW", "EKS", "IMP", "PKB", "NET_EXPORTS"]
small_fields = ["twKI", "twKS", "twINW", "twEKS", "twIMP"]

# Plot large-number indices using Matplotlib
plt.figure(figsize=(12, 6))
for field in large_fields:
    if field in data:
        plt.plot(years, data[field], marker='o', label=field)

plt.title("Indices with Large Numbers Over Time", fontsize=16)
plt.xlabel("Year", fontsize=14)
plt.ylabel("Value", fontsize=14)
plt.legend(loc='upper left', fontsize=10)
plt.grid(True)
plt.tight_layout()
plt.savefig("large_indices_plot.png")
plt.show()

# Plot small-number indices using Matplotlib
plt.figure(figsize=(12, 6))
for field in small_fields:
    if field in data:
        plt.plot(years, data[field], marker='o', label=field)

plt.ylim(0.8, 1.2)  # Adjust the Y-axis range for better visualization
plt.title("Indices with Small Numbers Over Time", fontsize=16)
plt.xlabel("Year", fontsize=14)
plt.ylabel("Value", fontsize=14)
plt.legend(loc='upper left', fontsize=10)
plt.grid(True)
plt.tight_layout()
plt.savefig("small_indices_plot.png")
plt.show()

# Plot ZDEKS on a separate graph using Matplotlib
if "ZDEKS" in data:
    plt.figure(figsize=(12, 6))
    plt.plot(years, data["ZDEKS"], marker='o', color='red', label="ZDEKS")

    plt.title("Export-to-GDP Ratio (ZDEKS) Over Time", fontsize=16)
    plt.xlabel("Year", fontsize=14)
    plt.ylabel("ZDEKS Value", fontsize=14)
    plt.legend(loc='upper left', fontsize=10)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig("zdeks_plot.png")
    plt.show()

# Write the updated results to the output JSON file
with open(output_path, 'w') as f:
    json.dump(data, f, indent=4)

print(f"Final results saved to {output_path}")